# Lab 3: Before/After Comparison

**Topic 2: Solving Overfitting Issues with Vibe Coding**

In this lab, we systematically compare four model variants to quantify the impact of different regularization strategies, visualize results side-by-side, and build an interactive prediction tool.

## Model Variants
1. **Baseline** -- Large dense network, no regularization
2. **Dropout Only** -- Baseline + Dropout layers
3. **Augmentation Only** -- CNN with data augmentation
4. **All Combined** -- CNN + Dropout + BatchNorm + Augmentation + L2 + Early Stopping

In [ ]:
# Run this cell in Google Colab to install dependencies
# Skip if running locally with uv
import sys
if 'google.colab' in sys.modules:
    !pip install -q keras torch torchvision gradio python-dotenv datasets transformers huggingface_hub
    print('Dependencies installed!')

## 1. Environment Setup

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"

import keras
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gradio as gr

print(f"Keras version: {keras.__version__}")
print(f"Keras backend: {keras.backend.backend()}")

## 2. Load and Prepare Data

In [ ]:
# Load Fashion-MNIST
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

# Normalize
x_train_full = x_train_full.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Split into train/val
x_train_raw = x_train_full[:50000]
x_val_raw = x_train_full[50000:]
y_train = y_train_full[:50000]
y_val = y_train_full[50000:]

# Flat version for dense models
x_train_flat = x_train_raw.reshape(-1, 784)
x_val_flat = x_val_raw.reshape(-1, 784)
x_test_flat = x_test.reshape(-1, 784)

# Image version for CNN models (with channel dimension)
x_train_img = x_train_raw.reshape(-1, 28, 28, 1)
x_val_img = x_val_raw.reshape(-1, 28, 28, 1)
x_test_img = x_test.reshape(-1, 28, 28, 1)

print(f"Training: {x_train_flat.shape} (flat), {x_train_img.shape} (image)")
print(f"Validation: {x_val_flat.shape}")
print(f"Test: {x_test_flat.shape}")

## 3. Define All Four Model Variants

In [ ]:
EPOCHS = 30
BATCH_SIZE = 128

def build_baseline():
    """Variant 1: Large dense network, no regularization."""
    model = keras.Sequential([
        keras.layers.Input(shape=(784,)),
        keras.layers.Dense(512, activation="relu"),
        keras.layers.Dense(512, activation="relu"),
        keras.layers.Dense(256, activation="relu"),
        keras.layers.Dense(256, activation="relu"),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dense(10, activation="softmax"),
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

def build_dropout_only():
    """Variant 2: Baseline + Dropout."""
    model = keras.Sequential([
        keras.layers.Input(shape=(784,)),
        keras.layers.Dense(512, activation="relu"),
        keras.layers.Dropout(0.5),
        keras.layers.Dense(512, activation="relu"),
        keras.layers.Dropout(0.5),
        keras.layers.Dense(256, activation="relu"),
        keras.layers.Dropout(0.4),
        keras.layers.Dense(256, activation="relu"),
        keras.layers.Dropout(0.4),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(10, activation="softmax"),
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

def build_augmentation_only():
    """Variant 3: CNN with data augmentation."""
    model = keras.Sequential([
        keras.layers.Input(shape=(28, 28, 1)),
        keras.layers.RandomFlip("horizontal"),
        keras.layers.RandomRotation(0.1),
        keras.layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Flatten(),
        keras.layers.Dense(256, activation="relu"),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dense(10, activation="softmax"),
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

def build_all_combined():
    """Variant 4: CNN + Dropout + BatchNorm + Augmentation + L2."""
    l2_reg = keras.regularizers.l2(1e-4)
    model = keras.Sequential([
        keras.layers.Input(shape=(28, 28, 1)),
        # Data augmentation
        keras.layers.RandomFlip("horizontal"),
        keras.layers.RandomRotation(0.1),
        # Conv block 1
        keras.layers.Conv2D(32, (3, 3), padding="same", kernel_regularizer=l2_reg),
        keras.layers.BatchNormalization(),
        keras.layers.Activation("relu"),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Dropout(0.25),
        # Conv block 2
        keras.layers.Conv2D(64, (3, 3), padding="same", kernel_regularizer=l2_reg),
        keras.layers.BatchNormalization(),
        keras.layers.Activation("relu"),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Dropout(0.25),
        # Dense layers
        keras.layers.Flatten(),
        keras.layers.Dense(256, kernel_regularizer=l2_reg),
        keras.layers.BatchNormalization(),
        keras.layers.Activation("relu"),
        keras.layers.Dropout(0.5),
        keras.layers.Dense(128, kernel_regularizer=l2_reg),
        keras.layers.BatchNormalization(),
        keras.layers.Activation("relu"),
        keras.layers.Dropout(0.4),
        keras.layers.Dense(10, activation="softmax"),
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

## 4. Train All Four Variants

In [ ]:
# Variant 1: Baseline
print("=" * 50)
print("Training Variant 1: Baseline (no regularization)")
print("=" * 50)
baseline_model = build_baseline()
baseline_history = baseline_model.fit(
    x_train_flat, y_train,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_data=(x_val_flat, y_val),
    verbose=1,
)

In [ ]:
# Variant 2: Dropout Only
print("=" * 50)
print("Training Variant 2: Dropout Only")
print("=" * 50)
dropout_model = build_dropout_only()
dropout_history = dropout_model.fit(
    x_train_flat, y_train,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_data=(x_val_flat, y_val),
    verbose=1,
)

In [ ]:
# Variant 3: Augmentation Only
print("=" * 50)
print("Training Variant 3: Augmentation Only")
print("=" * 50)
augmentation_model = build_augmentation_only()
augmentation_history = augmentation_model.fit(
    x_train_img, y_train,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_data=(x_val_img, y_val),
    verbose=1,
)

In [ ]:
# Variant 4: All Combined (with Early Stopping)
print("=" * 50)
print("Training Variant 4: All Combined")
print("=" * 50)
combined_model = build_all_combined()

early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True, verbose=1
)

combined_history = combined_model.fit(
    x_train_img, y_train,
    epochs=50,  # Higher limit; early stopping will halt
    batch_size=BATCH_SIZE,
    validation_data=(x_val_img, y_val),
    callbacks=[early_stopping],
    verbose=1,
)

## 5. Plot 2x2 Learning Curves Grid

In [ ]:
all_variants = {
    "Baseline (No Regularization)": baseline_history.history,
    "Dropout Only": dropout_history.history,
    "Augmentation Only": augmentation_history.history,
    "All Combined": combined_history.history,
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

colors = ["#e74c3c", "#3498db", "#2ecc71", "#9b59b6"]

for idx, (name, h) in enumerate(all_variants.items()):
    ax = axes[idx]
    epochs_range = range(1, len(h["loss"]) + 1)
    
    # Plot loss
    ax.plot(epochs_range, h["loss"], color=colors[idx], linestyle="-", label="Train Loss", linewidth=2)
    ax.plot(epochs_range, h["val_loss"], color=colors[idx], linestyle="--", label="Val Loss", linewidth=2)
    
    # Add accuracy on secondary y-axis
    ax2 = ax.twinx()
    ax2.plot(epochs_range, h["accuracy"], color="gray", linestyle="-", label="Train Acc", linewidth=1, alpha=0.6)
    ax2.plot(epochs_range, h["val_accuracy"], color="gray", linestyle="--", label="Val Acc", linewidth=1, alpha=0.6)
    ax2.set_ylabel("Accuracy", color="gray", fontsize=10)
    ax2.set_ylim([0.7, 1.0])
    ax2.tick_params(axis="y", labelcolor="gray")
    
    gap = h["accuracy"][-1] - h["val_accuracy"][-1]
    best_ep = np.argmin(h["val_loss"]) + 1
    ax.set_title(f"{name}\nGap: {gap:.3f} | Best Epoch: {best_ep}", fontsize=11, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss", color=colors[idx], fontsize=10)
    ax.tick_params(axis="y", labelcolor=colors[idx])
    ax.grid(True, alpha=0.3)
    
    # Combined legend
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc="upper right")

plt.suptitle("Before/After Comparison: Learning Curves for All Variants", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

## 6. Create Summary Table

In [ ]:
# Collect models and their info
models_info = [
    ("Baseline", baseline_model, baseline_history.history),
    ("Dropout Only", dropout_model, dropout_history.history),
    ("Augmentation Only", augmentation_model, augmentation_history.history),
    ("All Combined", combined_model, combined_history.history),
]

summary_data = []
for name, model, h in models_info:
    best_epoch = np.argmin(h["val_loss"]) + 1
    best_val_acc = h["val_accuracy"][best_epoch - 1]
    final_train_acc = h["accuracy"][-1]
    final_val_acc = h["val_accuracy"][-1]
    overfitting_gap = final_train_acc - final_val_acc
    total_params = model.count_params()
    epochs_trained = len(h["loss"])
    
    summary_data.append({
        "Model": name,
        "Best Val Accuracy": f"{best_val_acc:.4f}",
        "Final Train Acc": f"{final_train_acc:.4f}",
        "Final Val Acc": f"{final_val_acc:.4f}",
        "Overfitting Gap": f"{overfitting_gap:.4f}",
        "Best Epoch": best_epoch,
        "Epochs Trained": epochs_trained,
        "Parameters": f"{total_params:,}",
    })

summary_df = pd.DataFrame(summary_data)
summary_df = summary_df.set_index("Model")
print("\n" + "=" * 80)
print("MODEL COMPARISON SUMMARY")
print("=" * 80)
print(summary_df.to_string())
print("=" * 80)

In [ ]:
# Visual bar chart of overfitting gaps
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

model_names = [info[0] for info in models_info]
val_accs = [float(summary_data[i]["Best Val Accuracy"]) for i in range(4)]
gaps = [float(summary_data[i]["Overfitting Gap"]) for i in range(4)]

bar_colors = ["#e74c3c", "#3498db", "#2ecc71", "#9b59b6"]

ax1.bar(model_names, val_accs, color=bar_colors, edgecolor="black", linewidth=0.5)
ax1.set_ylabel("Best Validation Accuracy")
ax1.set_title("Validation Accuracy by Model Variant", fontsize=13)
ax1.set_ylim([0.85, 0.95])
ax1.grid(axis="y", alpha=0.3)
for i, v in enumerate(val_accs):
    ax1.text(i, v + 0.002, f"{v:.4f}", ha="center", fontsize=10, fontweight="bold")

ax2.bar(model_names, gaps, color=bar_colors, edgecolor="black", linewidth=0.5)
ax2.set_ylabel("Overfitting Gap (Train - Val Accuracy)")
ax2.set_title("Overfitting Gap by Model Variant", fontsize=13)
ax2.grid(axis="y", alpha=0.3)
for i, v in enumerate(gaps):
    ax2.text(i, v + 0.003, f"{v:.4f}", ha="center", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.show()

## 7. Evaluate All Models on Test Set

In [ ]:
print("Test Set Evaluation")
print("=" * 40)

# Baseline and Dropout use flat data; Augmentation and Combined use image data
test_inputs = [
    ("Baseline", baseline_model, x_test_flat),
    ("Dropout Only", dropout_model, x_test_flat),
    ("Augmentation Only", augmentation_model, x_test_img),
    ("All Combined", combined_model, x_test_img),
]

for name, model, test_x in test_inputs:
    test_loss, test_acc = model.evaluate(test_x, y_test, verbose=0)
    print(f"{name:25s} | Test Acc: {test_acc:.4f} | Test Loss: {test_loss:.4f}")

## 8. Gradio Interface: Model Variant Prediction Explorer

Select a model variant from the dropdown to see predictions and confidence on random test samples.

In [ ]:
# Store models in a dict for easy access
trained_models = {
    "Baseline": (baseline_model, "flat"),
    "Dropout Only": (dropout_model, "flat"),
    "Augmentation Only": (augmentation_model, "image"),
    "All Combined": (combined_model, "image"),
}

def predict_with_variant(variant_name):
    """Show predictions and confidence for a random batch of test images."""
    model, input_type = trained_models[variant_name]
    
    # Select 8 random test images
    indices = np.random.choice(len(x_test), 8, replace=False)
    
    if input_type == "flat":
        batch = x_test_flat[indices]
    else:
        batch = x_test_img[indices]
    
    predictions = model.predict(batch, verbose=0)
    true_labels = y_test[indices]
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    
    for i, ax in enumerate(axes.flatten()):
        # Show image
        ax.imshow(x_test[indices[i]], cmap="gray")
        
        pred_class = np.argmax(predictions[i])
        confidence = predictions[i][pred_class] * 100
        true_class = true_labels[i]
        
        color = "green" if pred_class == true_class else "red"
        ax.set_title(
            f"Pred: {class_names[pred_class]}\n"
            f"True: {class_names[true_class]}\n"
            f"Conf: {confidence:.1f}%",
            fontsize=9, color=color, fontweight="bold"
        )
        ax.axis("off")
    
    plt.suptitle(f"Model: {variant_name}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    
    # Create confidence bar chart for the first image
    fig2, ax2 = plt.subplots(figsize=(10, 4))
    probs = predictions[0] * 100
    bar_colors = ["#2ecc71" if j == true_labels[0] else "#3498db" for j in range(10)]
    bar_colors[np.argmax(predictions[0])] = "#e74c3c" if np.argmax(predictions[0]) != true_labels[0] else "#2ecc71"
    
    ax2.barh(class_names, probs, color=bar_colors, edgecolor="black", linewidth=0.5)
    ax2.set_xlabel("Confidence (%)")
    ax2.set_title(f"Confidence Distribution (First Image) - {variant_name}", fontsize=12)
    ax2.set_xlim([0, 100])
    ax2.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    
    # Compute accuracy on a larger sample
    if input_type == "flat":
        full_preds = model.predict(x_test_flat, verbose=0)
    else:
        full_preds = model.predict(x_test_img, verbose=0)
    
    pred_classes = np.argmax(full_preds, axis=1)
    accuracy = np.mean(pred_classes == y_test) * 100
    correct = np.sum(pred_classes == y_test)
    
    info_text = (
        f"Model: {variant_name}\n"
        f"Test Accuracy: {accuracy:.2f}%\n"
        f"Correct: {correct} / {len(y_test)}\n"
        f"Incorrect: {len(y_test) - correct} / {len(y_test)}"
    )
    
    return fig, fig2, info_text

demo = gr.Interface(
    fn=predict_with_variant,
    inputs=gr.Dropdown(
        choices=["Baseline", "Dropout Only", "Augmentation Only", "All Combined"],
        value="Baseline",
        label="Select Model Variant",
    ),
    outputs=[
        gr.Plot(label="Predictions on Random Test Images"),
        gr.Plot(label="Confidence Distribution (First Image)"),
        gr.Textbox(label="Model Performance", lines=5),
    ],
    title="Before/After Model Comparison",
    description="Select a model variant to see its predictions on random Fashion-MNIST test images. Compare how different regularization strategies affect prediction confidence and accuracy.",
)

demo.launch()